In [1]:
!pip install jupyter_black


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%load_ext jupyter_black

In [3]:
# read in all the words
words = open("data/names.txt", "r").read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [5]:
# build the dataset
block_size = 3


def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]  # crop and append

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y


import random

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])  # 80%
Xdev, Ydev = build_dataset(words[n1:n2])  # 10%
Xte, Yte = build_dataset(words[n2:])  # 10%

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [82]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
    assert dt.shape == t.grad.shape, f"{dt.shape=}, {t.grad.shape=}"
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(
        f"{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}"
    )

In [7]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1 # using b1 just for fun, it's useless because of BN
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

# Note: I am initializating many of these parameters in non-standard ways
# because sometimes initializating with e.g. all zeros could mask an incorrect
# implementation of the backward pass.

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

4137


In [8]:
batch_size = 32
n = batch_size  # a shorter variable also, for convenience
# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]  # batch X,Y

In [9]:
emb = C[Xb]  # (26, 10)
embcat = emb.view(emb.shape[0], -1)

emb.shape, embcat.shape

(torch.Size([32, 3, 10]), torch.Size([32, 30]))

In [10]:
# forward

emb = C[Xb]  # (batch_size, 3, 10)
embcat = emb.view(emb.shape[0], -1)  # (bs, 30)
# layer 1
z1 = embcat @ W1 + b1
# batchnorm
bnmeani = 1 / n * z1.sum(0, keepdim=True)
bndiff = z1 - bnmeani
bndiff2 = bndiff**2
bnvar = 1 / (n - 1) * bndiff2.sum(0, keepdim=True)
bnvar_inv = (bnvar + 1e-5) ** -0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-linearity
h = torch.tanh(hpreact)  # hidden layer
print(h.shape)
# Linear layer 2
logits = h @ W2 + b2  # output layer
print(logits.shape)

# loss (cross entropy)
# logits_max = logits.max(1, keepdims=True) # return return type torch.return_type.max
# print(logits_max)
logits_max = logits.max(1, keepdims=True).values
norm_logits = logits - logits_max  # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = (
    counts_sum**-1
)  # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = torch.log(probs)
loss = -logprobs[range(n), Yb].mean()
print(loss)

# PyTorch backward pass
for p in parameters:
    p.grad = None
for t in [
    logprobs,
    probs,
    counts,
    counts_sum,
    counts_sum_inv,  # afaik there is no cleaner way
    norm_logits,
    logits_max,
    logits,
    h,
    hpreact,
    bnraw,
    bnvar_inv,
    bnvar,
    bndiff2,
    bndiff,
    z1,
    bnmeani,
    embcat,
    emb,
]:
    t.retain_grad()
loss.backward()
loss

torch.Size([32, 64])
torch.Size([32, 27])
tensor(3.3298, grad_fn=<NegBackward0>)


tensor(3.3298, grad_fn=<NegBackward0>)

### imp 

loss = -logprobs[range(n), Yb].mean()

logprobs.shape -> (32, 27)

range(n) -> for each row

take the Yb meaning real class others will be 0

hence the it will be prob of that index .mean()

In [138]:
# backward
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0 / n
cmp("logprobs", dlogprobs, logprobs)
dprobs = (1.0 / probs) * dlogprobs
cmp("probs", dprobs, probs)
dcounts_sum_inv = (counts * dprobs).sum(1, keepdims=True)
cmp("counts_sum_ing", dcounts_sum_inv, counts_sum_inv)
dcounts_sum = -(counts_sum**-2) * dcounts_sum_inv
cmp("counts_sum", dcounts_sum, counts_sum)
dcounts = (
    dcounts_sum.clone() * torch.ones_like(counts) + counts_sum_inv * dprobs
)  ## here we could have skip *torch.ones_like(counts) cause broadcasting would have handld but bee explicit
cmp("counts", dcounts, counts)
dnorm_logits = norm_logits.exp() * dcounts  # counts = norm_logits.exp()
cmp("norm_logits", dnorm_logits, norm_logits)
dlogits_max = -dnorm_logits.sum(1, keepdims=True)
cmp("logits_max", dlogits_max, logits_max)
ones_in_logits = torch.zeros_like(logits)
ones_in_logits[range(n), torch.argmax(logits, dim=1)] = 1.0
dlogits = dnorm_logits.clone() + ones_in_logits * dlogits_max
cmp("logits", dlogits, logits)
dh = dlogits @ W2.T
cmp("h", dh, h)
dW2 = h.T @ dlogits
cmp("W2", dW2, W2)
db2 = dlogits.sum(0)
cmp("db2", db2, b2)
dhpreact = (1 - h**2) * dh
cmp("hpreact", dhpreact, hpreact)
dbngain = (bnraw * dhpreact).sum(0, keepdims=True)
cmp("bngain", dbngain, bngain)
dbnraw = bngain * dhpreact
cmp("bnraw", dbnraw, bnraw)
dbnbias = dhpreact.sum(0, keepdims=True)
cmp("bnbias", dbnbias, bnbias)
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdims=True)
cmp("bnvar_inv", dbnvar_inv, bnvar_inv)
dbnvar = -0.5 * ((bnvar + 1e-5) ** -1.5) * dbnvar_inv
cmp("dbnvar", dbnvar, bnvar)
dbndiff2 = torch.ones_like(bndiff2) * (
    dbnvar / (n - 1)
)  # same but minor diff i.e 1e-12
# dbndiff2 = (1.0 / (n - 1)) * torch.ones_like(bndiff2) * dbnvar
cmp("dbndiff2", dbndiff2, bndiff2)
dbndiff = bnvar_inv * dbnraw + 2 * bndiff * dbndiff2
cmp("bndiff", dbndiff, bndiff)
dbnmeani = -dbndiff.sum(0, keepdims=True)
cmp("bnmeani", dbnmeani, bnmeani)
dz1 = (1.0 / n) * dbnmeani + dbndiff
cmp("z1", dz1, z1)
dembcat = dz1 @ W1.T
cmp("embcat", dembcat, embcat)
dW1 = embcat.T @ dz1
cmp("W1", dW1, W1)
db1 = dz1.sum(0)
cmp("b1", db1, b1)
demb = dembcat.view(emb.shape)
cmp("emb", demb, emb)
dC = torch.zeros_like(C)
for i in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        idx = Xb[i, j]
        # print(idx)
        dC[idx] += demb[i, j]

cmp("C", dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_ing  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logits_max      | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
db2             | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff:

In [23]:
torch.ones_like(counts).shape, dcounts_sum.shape

(torch.Size([32, 27]), torch.Size([32, 1]))

In [50]:
dnorm_logits.shape, ones_in_logits.shape

(torch.Size([32, 27]), torch.Size([32, 27]))

In [115]:
W1.shape, dz1.shape, embcat.shape

(torch.Size([30, 64]), torch.Size([32, 64]), torch.Size([32, 30]))

In [127]:
dC.shape, demb.shape

(torch.Size([27, 10]), torch.Size([32, 3, 10]))